In [52]:
import pandas as pd
import numpy as np

# Importacion de Datos recuperados de LOKI

In [66]:
df = pd.read_csv("../datasets/01_logs_dataset.csv")
df.head()

,@timestamp,service,level,event_type,outcome,http_method,http_uri,http_status,duration_ms,error_type,error_message,error_origin,traceId
0,2026-07-03T08:03:23.032Z,api-autenticacion,INFO,HTTP_REQUEST,SUCCESS,POST,/api/auth/register,201.0,363.0,NaN,NaN,NaN,6a476cca227cf2c2a7778eea6814c767
1,2026-07-03T08:04:56.876Z,api-autenticacion,INFO,HTTP_REQUEST,SUCCESS,POST,/api/auth/login,200.0,115.0,NaN,NaN,NaN,6a476d28bef605f56445e5f31a75ab8a
2,2026-07-03T08:04:59.767Z,api-autenticacion,INFO,HTTP_REQUEST,SUCCESS,POST,/api/auth/login,200.0,108.0,NaN,NaN,NaN,6a476d2b308ea5cc0f8932e9de3ca02d
3,2026-07-03T08:05:04.616Z,api-autenticacion,INFO,HTTP_REQUEST,SUCCESS,POST,/api/auth/login,200.0,107.0,NaN,NaN,NaN,6a476d30b15c719121fc2c7488b11b02
4,2026-07-03T08:05:06.940Z,api-autenticacion,INFO,HTTP_REQUEST,SUCCESS,POST,/api/auth/login,200.0,87.0,NaN,NaN,NaN,6a476d323360f89f718e7343d5f6686b


In [67]:
print(f"Total de registros: {len(df)}")
print(f"\nColumnas disponibles:\n{df.columns.tolist()}")
print(f"\nDistribución de event_type:\n{df['event_type'].value_counts()}")
print(f"\nDistribución de level:\n{df['level'].value_counts()}")
print(f"\nDistribución de service:\n{df['service'].value_counts()}")
print(f"\nValores nulos por columna:\n{df.isnull().sum()}")

Total de registros: 4970

Columnas disponibles:
['@timestamp', 'service', 'level', 'event_type', 'outcome', 'http_method', 'http_uri', 'http_status', 'duration_ms', 'error_type', 'error_message', 'error_origin', 'traceId']

Distribución de event_type:
event_type
HTTP_REQUEST       1145
UNHANDLED_ERROR    1115
SERVER_ERROR        966
Name: count, dtype: int64

Distribución de level:
level
ERROR    2389
WARN     1349
INFO     1232
Name: count, dtype: int64

Distribución de service:
service
api-inventario       3081
api-pedidos          1834
api-autenticacion      55
Name: count, dtype: int64

Valores nulos por columna:
@timestamp          0
service             0
level               0
event_type       1744
outcome          2859
http_method      2859
http_uri         2859
http_status      2859
duration_ms      2859
error_type       3855
error_message    4051
error_origin     4465
traceId            98
dtype: int64


## Documentacion sobre los datos
- *event_type, service, http_metos*: Son datos ordinales. Nos indican el tipo de peticion. Usarmeos OnehotEncoder
- *duration_ms, http_status*: Valores numericos que se pueden utilizar de entrada. Podria usarse StandardScaler para mejorar la precision final
- *@timestamp*: De tipo temporal. Requiere de un procesamiento donde vamos a sacar varias columnas: hour_of_day, day_of_week, minute_of_hour
- *level*: es de categoria ordinal, es decir, Warn, Info y Error, tendrán pesos diferentes e indicaran diferentes grados, en este caso, de gravedad. Por lo tanto se mapearan a 0, 1, 2 para que el modelo comprenda que contienen diferentes importancias
- *error_message, error_origin*: Serán tratados por un NLP en un futuro. Por ahora trataremos error_message como un valor de 0, si existe o 1, sino existe
- *outcome*: Aporta redundancia al dataset, ya tenemos http_status para que el modelo pueda detectar si la operacion ha salido correctaamente o de manera fallida. Se elimina
- *traceId*: Cambiante para cada transaccion, el modelo tenderia a memorizar. Esta variable se eleminar porque confundiria al modelo

# Limpieza de valores nulos y anomalías

In [68]:
def nulos(df):
    nulos = df.isna().sum()
    columnas_nulas = nulos[nulos>0]
    print(columnas_nulas.sort_values(ascending=False))

In [56]:
df = df.drop(columns=["outcome"])

In [69]:
# Elimina filas donde event_type, http_uri y duration_ms son todos nulos
df = df.dropna(subset=["event_type", "http_uri", "duration_ms"], how="all")

# Elimina también los endpoints de caos
df = df[~df["http_uri"].str.contains("/chaos/", na=False)]

In [70]:
nulos(df)

error_origin     2139
error_message    1725
error_type       1529
outcome          1115
http_method      1115
http_uri         1115
http_status      1115
duration_ms      1115
dtype: int64


- *duration_ms*: sacar la media puede ser peligroso, por lo que decido actuar con la media
- *error_type*: cambiar a NONE aquellas que estan en NAN para conseguir que el modelo entienda que no hay errores cuando es sucess

In [76]:
# MOSTRAR SALIDAS DE ERROR
salidas_error = cdf[cdf["http_status"] != 200]
salidas_error

,@timestamp,service,level,event_type,outcome,http_method,http_uri,http_status,duration_ms,error_type,traceId,error_message
0,2026-07-03T08:03:23.032Z,api-autenticacion,INFO,HTTP_REQUEST,SUCCESS,POST,/api/auth/register,201.0,363.0,NaN,6a476cca227cf2c2a7778eea6814c767,NaN
1140,2026-07-03T08:05:35.850Z,api-inventario,ERROR,SERVER_ERROR,FAILURE,GET,/inventario,500.0,30060.0,NaN,6a476d3147869f889d92defc1f2c96fe,NaN
1141,2026-07-03T08:05:35.850Z,api-inventario,ERROR,SERVER_ERROR,FAILURE,GET,/inventario,500.0,30060.0,NaN,6a476d3147869f889d92defc1f2c96fe,Could not open JPA EntityManager for transaction
1142,2026-07-03T08:05:36.720Z,api-inventario,ERROR,SERVER_ERROR,FAILURE,POST,/inventario/1/retirar,500.0,30023.0,NaN,6a476d328b5403ed9dcd89454f2cfbc4,NaN
1143,2026-07-03T08:05:36.720Z,api-inventario,ERROR,SERVER_ERROR,FAILURE,POST,/inventario/1/retirar,500.0,30023.0,NaN,6a476d328b5403ed9dcd89454f2cfbc4,Could not open JPA EntityManager for transaction
...,...,...,...,...,...,...,...,...,...,...,...,...
3392,2026-07-03T08:40:52.592Z,api-pedidos,ERROR,SERVER_ERROR,FAILURE,POST,/pedidos,500.0,5022.0,NaN,6a47758ffac02a9e4213db9585b587a7,I/O error on POST request for 'http://api-inve...
3393,2026-07-03T08:40:52.706Z,api-pedidos,ERROR,SERVER_ERROR,FAILURE,POST,/pedidos,500.0,5029.0,NaN,6a47758f0ede2cc79bdcd4b0f2711f8d,NaN
3394,2026-07-03T08:40:52.706Z,api-pedidos,ERROR,SERVER_ERROR,FAILURE,POST,/pedidos,500.0,5029.0,NaN,6a47758f0ede2cc79bdcd4b0f2711f8d,Could not open JPA EntityManager for transaction
3395,2026-07-03T08:40:52.706Z,api-pedidos,ERROR,SERVER_ERROR,FAILURE,POST,/pedidos,500.0,5029.0,NaN,6a47758f0ede2cc79bdcd4b0f2711f8d,NaN


In [45]:
# === DEJAMOS ESTO EN MANOS DEL INPUTER

# df["duration_ms"] = df["duration_ms"].fillna(df["duration_ms"].median())
# df["error_type"] = df["error_type"].fillna("NONE")
# df["http_method"] = df["http_method"].fillna("UNKNOWN")
# df["http_status"] = df["http_status"].fillna(0)

In [75]:
# Logs del filtro HTTP
df["http_uri"] = df["http_uri"].fillna("UNKNOWN")

filtro = df[df["http_uri"] != "UNKNOWN"].copy()

# Logs de error
errores = df[df["error_type"] != "NONE"][["traceId", "error_type", "error_origin", "error_message"]].copy()

# Une por traceId — una fila por petición con toda la información
df_completo = filtro.merge(errores, on="traceId", how="left")
df_completo = df_completo.rename(columns={"error_type_x": "error_type", "error_message_y": "error_message"})
cdf = df_completo.drop(columns=["error_origin_x", "error_origin_y", "error_type_y", "error_message_x"])
print(f"Dataset unificado: {len(df_completo)} filas")

Dataset unificado: 3397 filas


In [77]:
print("Nulos:")
nulos(cdf)
print("=============")
print(f"Numero de filas que son errores: {len(errores)}")
print(f"Numero de filas totales {len(cdf)}")

Nulos:
error_type       3397
error_message    2147
dtype: int64
Numero de filas que son errores: 2644
Numero de filas totales 3397


In [78]:
def uri_a_servicio_destino(uri):
    if "/inventario" in uri: return "inventario"
    if "/pedidos" in uri: return "pedidos"
    if "/auth" in uri: return "autenticacion"
    return "unknown"

    
# aplicamos los cambios
cdf["error_message_binary"] = cdf["error_message"].notna().astype(int)
cdf["uri_servicio"] = cdf["http_uri"].apply(uri_a_servicio_destino)

# eliminamos las columnas que ya no sirven
cdf = cdf.drop(columns=["http_uri", "error_message"])
print(cdf["uri_servicio"].value_counts())
print("----------")
print(cdf["error_message_binary"].value_counts())

uri_servicio
inventario       1917
pedidos          1459
autenticacion      21
Name: count, dtype: int64
----------
error_message_binary
0    2147
1    1250
Name: count, dtype: int64


## Transformacion de datos y escalado

In [79]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder

# event_type, service, http_metod, http_status -> one hot encoding
categorical_scaler = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='constant', fill_value='NONE')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# duration_ms -> Standar scaler
num_scaler = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# level -> OrdinalEncodera
level_pipeline = Pipeline(steps=[
    ('inputer', SimpleImputer(strategy='constant',fill_value='INFO')),
    ('ordinal', OrdinalEncoder(categories=[["INFO", "WARN", "ERROR"]]))
])


# Clase para descpomponer timestamp
class TimestampDescomponedor(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self  
    
    def transform(self, X):
        # Convierte a Series independientemente de si llega como DataFrame, array o Series
        if hasattr(X, 'iloc'):
            # Es un DataFrame — coge la primera columna
            serie = pd.to_datetime(X.iloc[:, 0])
        else:
            # Es un array numpy
            serie = pd.to_datetime(pd.Series(X.ravel()))
        
        resultado = pd.DataFrame({
            "hour_of_day":    serie.dt.hour,
            "day_of_week":    serie.dt.dayofweek,
            "minute_of_hour": serie.dt.minute
        })
        
        return resultado.values
timestamp_pipeline = Pipeline(steps=[
    ("descomponedor", TimestampDescomponedor()),
    ("scaler", StandardScaler())
])

In [80]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_scaler, ["event_type", "service", "http_method", "uri_servicio", "http_status"]),
    ("num", num_scaler, ["duration_ms"]),
    ("ts", timestamp_pipeline, ["@timestamp"]),
    ("ord", level_pipeline, ["level"])
])

# -------
### GridSearch, Train test split, train

In [81]:
from sklearn.model_selection import train_test_split, GridSearchCV

# para evitar data leakage, calculamos sobre los dataframes separados la variable de supervisado
df_train, df_test = train_test_split(cdf, test_size=0.2, random_state=42)

umbral_tiempo_anomalo = df_train["duration_ms"].quantile(0.95)
print(f"Umbral de lentitud fijado en: {umbral_tiempo_anomalo:.2f} ms")

def etiquetar_anomalias(dataset, umbral_tiempo):
    return (
        (dataset["level"] == "ERROR") |
        (dataset["event_type"].isin(["SERVER_ERROR", "UNHANDLED_ERROR"])) |
        (dataset["duration_ms"] > umbral_tiempo)
    ).astype(int)

df_train["es_anomalia"] = etiquetar_anomalias(df_train, umbral_tiempo_anomalo)
df_test["es_anomalia"]  = etiquetar_anomalias(df_test, umbral_tiempo_anomalo)


Umbral de lentitud fijado en: 30021.00 ms


In [98]:
# === datos de train y test ===
X_train = df_train.drop(columns=["es_anomalia"])
y_train = df_train["es_anomalia"]

X_test = df_test.drop(columns=["es_anomalia"])
y_test = df_test["es_anomalia"]